# Entendimiento de los Datos

El siguiente notebook compende extracción de los datos desde las fuentes preparadas en edunautica.mx para su inicial comprensión.

Este notebook sirve para poder explorar como estan compuestos, conocer los tipos de datos, cantidad y retos que se enfrentarán en cara al análisis de los datos con procesos como el EDA.

Al principio del proyecto, la idea era ofrecer a edunautica.mx un marco de trabajo con rigor estadístico para adoptar estrategias de adquisición de leads con rigor estadistico que les permitira proyectar costos y optimizar adquisición de Leads dentro de su producto digital.

Sin embargo, durante el analisis de datos, pude observar que, existia un considerable número de tráfico automático, por lo que moví el proyecto hacia un problema de clasificación de machine learning, dado que cada día reciben más de 80 mil eventos en sus logs de navegación. Esto se puede apreciar mejor en la parte 02 de los notebooks.

In [1]:
from supabase import create_client, Client
import os
from pathlib import Path
import sys
import pandas as pd
from google.oauth2 import service_account
import json
from google.cloud import bigquery
import numpy as np
import geopandas as gpd


PROJECT_ROOT = Path.cwd().parent
SRC = PROJECT_ROOT / 'src'

if str(SRC) not in sys.path:
    sys.path.insert(0, str(SRC))

from utils.gather import (
    extract_and_save_data,
    load_data_from_table
)

from utils.cleaner import (
    limpiar_geodatos
) 

SUPABASE_URL = os.environ.get('SUPABASE_URL')
SUPABASE_KEY = os.environ.get('SUPABASE_KEY')

supabase: Client = create_client(SUPABASE_URL, SUPABASE_KEY)

file_service_route = os.environ.get('URL_GOOGLE_AUTH')

with open(file_service_route, 'r', encoding='utf-8') as jsonfile:
    service_account_info = json.load(jsonfile)

credenciales = service_account.Credentials.from_service_account_info(service_account_info)

SAVING_ROUTE_DATA = PROJECT_ROOT / 'Data' / 'csv'

proyecto = 'datos_edunautica'
dataset = 'analytics_470300720'

download_data = False

db_path = PROJECT_ROOT / 'Data' / 'db_producto' / 'base_datos'

df_group = []

niveles = ['educacion_inicial_privadas',
           'educacion_preescolar_privadas',
           'educacion_primaria_privadas',
           'educacion_preparatoria_privadas',
           'educacion_secundaria_privadas']

for nivel in niveles:
  nombre_path = os.path.join(db_path, nivel+('.csv'))
  df = pd.read_csv(nombre_path)
  df = df.drop(columns='Unnamed: 0')
  df_group.append(df)

PROCESSED_ROUTE = PROJECT_ROOT / 'Data' / 'processed'
BIG_QUERY_DATA = PROJECT_ROOT / 'Data' / 'big_query'
VERCEL_DRAINS = PROJECT_ROOT / 'Data' / 'vercel_drains'

URL_BUCKET = './'


## Descarga de Datos desde SUPABASE (Deprecado)

Este es el proceso de carga de datos de supabase, necesita variables de entorno configuradas para funcionar y haber creado la tabla vercel_logs_buffer para descargar los logdrains de producción.

Este proceso esta pensado para menos de 100 mil registros de la base de datos, ahora mismo la tabla de vercel logs contiene alrededor de 400 mil registros, por lo que tome la desición de removerlo, pero dejarlo documentado para fines académicos.

En pasos posteriores utilizo, ventanas de 5 minutos usando psql y para los datos completos de entrenamiento utilizo psql con una exportación hacia archivos csv fuera de python.

Por ello utilizo un archivo que esta almacenado en una carpeta llamada Data, fuera del repositorio del proyecto dada la sensibilidad de los datos.

In [2]:
"""
nombre_tabla = 'vercel_logs_buffer'

print('Este es un proceso en base de datos en supabase, deberia tardar un par de minutos.')
sample = load_data_from_table(nombre_tabla, supabase)
sample_frame = pd.DataFrame(sample)

log_data = pd.json_normalize(sample_frame['log'])

print('Este sample, es una muestra de trabajo con datos controlados para fines de entrega del TEC')
file_name = 'sample_20260528.parquet'
log_data.to_parquet(os.path.join(VERCEL_DRAINS, file_name))
"""

"\nnombre_tabla = 'vercel_logs_buffer'\n\nprint('Este es un proceso en base de datos en supabase, deberia tardar un par de minutos.')\nsample = load_data_from_table(nombre_tabla, supabase)\nsample_frame = pd.DataFrame(sample)\n\nlog_data = pd.json_normalize(sample_frame['log'])\n\nprint('Este sample, es una muestra de trabajo con datos controlados para fines de entrega del TEC')\nfile_name = 'sample_20260528.parquet'\nlog_data.to_parquet(os.path.join(VERCEL_DRAINS, file_name))\n"

## Procesamiento de Datos para Analytics de Negocio

Esta es la carga de datos y una limpieza que necesitaban las coordenadas de la DB.

Las siguientes celdas corresponden a analisis exploratorio de datos para comprender donde existen incidencias. Es importante mencionarlo ya que la idea de encontrar y clasificar userAgents ha nacido desde este análisis.

In [3]:
megaframe = pd.concat(df_group, ignore_index = True)

megaframe = limpiar_geodatos(megaframe,  ## Mapa de Escuelas de todos los niveles
                             col_lat='latitud',
                             col_lon='longitud')

megageo = gpd.GeoDataFrame(megaframe, ## DB de todas las escuelas de nivel inicial a prepa
                           geometry = gpd.points_from_xy(megaframe.longitud,
                                                         megaframe.latitud),
                           crs = 'EPSG:4326')

Filas eliminadas: 354


In [4]:
if download_data:
    print('La data de supabase sera descargada: \n')
    table_name = 'busquedas_escolares'
    busquedas_mapa = extract_and_save_data(supabase, 'busquedas_escolares', 'busquedas_escolares.csv')

    table_name = 'edunautica_leads'
    leads = extract_and_save_data(supabase, table_name, f'{table_name}.csv')

    table_name = 'edunautica_tracking'
    tracking_server_side = extract_and_save_data(supabase, table_name, f'{table_name}.csv')
else:
    print('La data de supabase sera precargada')
    file_route = os.path.join(SAVING_ROUTE_DATA, 'busquedas_escolares.csv')
    busquedas = pd.read_csv(
        file_route,
        dtype= {
            'ga_client_id' : 'str'
        }
)

    file_route = os.path.join(SAVING_ROUTE_DATA, 'edunautica_leads.csv')
    leads = pd.read_csv(file_route)

    file_route = os.path.join(SAVING_ROUTE_DATA, 'edunautica_tracking.csv')
    tracking = pd.read_csv(file_route)

La data de supabase sera precargada


In [5]:
search_console_path = BIG_QUERY_DATA / 'search_console_data.csv'

print('Este será un proceso pesado, puede tardar hasta 3 minutos.')
search_console = pd.read_csv(search_console_path)
search_console['cct_normalized'] = search_console['url'].str[-10:].str.lower()
search_console['nivel'] = search_console['url'].str.split('/').str[3]
search_console['entidad'] = search_console['url'].str.split('/').str[4]
search_console['municipio'] = search_console['url'].str.split('/').str[5]
search_console['escuela'] = search_console['url'].str.split('/').str[6]

Este será un proceso pesado, puede tardar hasta 3 minutos.


In [6]:
search_console['data_date'] = pd.to_datetime(search_console['data_date'])
search_console.to_csv(str(PROCESSED_ROUTE) + '/search_console.csv', index = False)

In [7]:
tracking['nivel'] = tracking['url_ref'].str.split('/').str[3]
tracking['entidad'] = tracking['url_ref'].str.split('/').str[4]
tracking['muicipio'] = tracking['url_ref'].str.split('/').str[5]
tracking['nombre_escuela'] = tracking['url_ref'].str.split('/').str[6]
tracking['cct'] = tracking['url_ref'].str[-10:]
tracking = tracking.rename(columns={'id_sesion' : 'id'})

complete_leads = leads.merge(
    tracking,
    on = 'id'
)

complete_leads.to_csv(str(PROCESSED_ROUTE) + '/leads_complete.csv', index = False)
tracking.to_csv(str(PROCESSED_ROUTE) + '/tracking_preprocessed.csv', index = False)

In [8]:
megaframe['cct_normalized'] = megaframe['cct'].str.lower()
busquedas_privadas = busquedas[busquedas['tipo'] == 'privadas']
busquedas_privadas['user_pseudo_id_str'] = busquedas_privadas['ga_client_id'].astype(str)

/var/folders/4b/cg1436q565d9rshq6hxb1dqc0000gn/T/ipykernel_41326/3289569352.py:3: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  busquedas_privadas['user_pseudo_id_str'] = busquedas_privadas['ga_client_id'].astype(str)


In [9]:
big_file_route = os.path.join(BIG_QUERY_DATA, '20260101_20260519.csv')
big_query = pd.read_csv(
    big_file_route,
    dtype = {
        'user_pseudo_id' : 'string' ## Nunca olvidar
    }
)
big_query['user_pseudo_id_str'] = big_query['user_pseudo_id'].astype(str)

In [10]:
big_query[big_query['user_pseudo_id_str'] == '698375144.1777477605']

,event_date,event_timestamp,event_name,user_pseudo_id,school_path,tipo,nivel,cta_placement,surface,page_location,cct,route_path,device,ga_session_id,ga_session_number,batch_page_id,user_pseudo_id_str
414,20260511,1778511141387561,click_school,698375144.1777477605,/primaria/guanajuato/leon/hilario-medina-11dpr...,home,primaria,desktop_list,map_results,https://www.edunautica.mx/,11DPR3916P,/,desktop,1778511116,7,1778511116244,698375144.1777477605
547,20260429,1777481729373947,click_school,698375144.1777477605,/primaria/guanajuato/leon/profr-luis-chavez-or...,home,primaria,desktop_list,map_results,https://www.edunautica.mx/primaria/guanajuato/...,11DPR0950B,/primaria/guanajuato/leon,desktop,1777481722,2,1777478553593,698375144.1777477605
886,20260505,1777991549900850,click_school,698375144.1777477605,/primaria/guanajuato/leon/benito-juarez-11dpr0...,home,primaria,desktop_list,map_results,https://www.edunautica.mx/primaria/guanajuato/...,11DPR0702U,/primaria/guanajuato/leon,desktop,1777991512,4,1777991512108,698375144.1777477605
887,20260505,1777991622800397,click_school,698375144.1777477605,/primaria/guanajuato/leon/lic-juan-jose-torres...,home,primaria,desktop_list,map_results,https://www.edunautica.mx/primaria/guanajuato/...,11EPR0700V,/primaria/guanajuato/leon,desktop,1777991512,4,1777991512108,698375144.1777477605
888,20260505,1777991659965660,click_school,698375144.1777477605,/primaria/guanajuato/leon/quetzalcoatl-11epr0486u,home,primaria,desktop_list,map_results,https://www.edunautica.mx/primaria/guanajuato/...,11EPR0486U,/primaria/guanajuato/leon,desktop,1777991512,4,1777991512108,698375144.1777477605
889,20260505,1777991704679462,click_school,698375144.1777477605,/primaria/guanajuato/leon/juan-escutia-11dpr2770e,home,primaria,desktop_list,map_results,https://www.edunautica.mx/primaria/guanajuato/...,11DPR2770E,/primaria/guanajuato/leon,desktop,1777991512,4,1777991512108,698375144.1777477605
890,20260505,1777991742197178,click_school,698375144.1777477605,/primaria/guanajuato/leon/benito-juarez-11dpr0...,home,primaria,desktop_list,map_results,https://www.edunautica.mx/primaria/guanajuato/...,11DPR0702U,/primaria/guanajuato/leon,desktop,1777991512,4,1777991512108,698375144.1777477605
891,20260505,1777991755080776,click_school,698375144.1777477605,/primaria/guanajuato/leon/lic-juan-jose-torres...,home,primaria,desktop_list,map_results,https://www.edunautica.mx/primaria/guanajuato/...,11EPR0700V,/primaria/guanajuato/leon,desktop,1777991512,4,1777991512108,698375144.1777477605
892,20260505,1777991775324833,click_school,698375144.1777477605,/primaria/guanajuato/leon/quetzalcoatl-11epr0486u,home,primaria,desktop_list,map_results,https://www.edunautica.mx/primaria/guanajuato/...,11EPR0486U,/primaria/guanajuato/leon,desktop,1777991512,4,1777991512108,698375144.1777477605
893,20260505,1777991791247299,click_school,698375144.1777477605,/primaria/guanajuato/leon/juan-escutia-11dpr2770e,home,primaria,desktop_list,map_results,https://www.edunautica.mx/primaria/guanajuato/...,11DPR2770E,/primaria/guanajuato/leon,desktop,1777991512,4,1777991512108,698375144.1777477605


In [11]:
busquedas_unicas = busquedas_privadas.drop_duplicates(subset='user_pseudo_id_str').merge(
    big_query,
    on = 'user_pseudo_id_str',
    how = 'inner'
)



busquedas_unicas.to_csv(str(PROCESSED_ROUTE) + '/busquedas_unicas.csv', index = False)

In [12]:
busquedas_unicas[busquedas_unicas['user_pseudo_id_str'] =='698375144.1777477605']

,search_id,created_at,search_source,nivel_x,tipo_x,route_path_x,cct_context,center_lat,center_lon,radius_m,...,nivel_y,cta_placement,surface,page_location,cct,route_path_y,device,ga_session_id_y,ga_session_number,batch_page_id
632,c68fc102-9a70-495a-a9ab-caef9be3fb08,2026-05-11T14:52:41.967748+00:00,postal_code,primaria,privadas,/,NaN,21.101535,-101.583732,1500,...,primaria,desktop_list,map_results,https://www.edunautica.mx/,11DPR3916P,/,desktop,1778511116,7,1778511116244
633,c68fc102-9a70-495a-a9ab-caef9be3fb08,2026-05-11T14:52:41.967748+00:00,postal_code,primaria,privadas,/,NaN,21.101535,-101.583732,1500,...,primaria,desktop_list,map_results,https://www.edunautica.mx/primaria/guanajuato/...,11DPR0950B,/primaria/guanajuato/leon,desktop,1777481722,2,1777478553593
634,c68fc102-9a70-495a-a9ab-caef9be3fb08,2026-05-11T14:52:41.967748+00:00,postal_code,primaria,privadas,/,NaN,21.101535,-101.583732,1500,...,primaria,desktop_list,map_results,https://www.edunautica.mx/primaria/guanajuato/...,11DPR0702U,/primaria/guanajuato/leon,desktop,1777991512,4,1777991512108
635,c68fc102-9a70-495a-a9ab-caef9be3fb08,2026-05-11T14:52:41.967748+00:00,postal_code,primaria,privadas,/,NaN,21.101535,-101.583732,1500,...,primaria,desktop_list,map_results,https://www.edunautica.mx/primaria/guanajuato/...,11EPR0700V,/primaria/guanajuato/leon,desktop,1777991512,4,1777991512108
636,c68fc102-9a70-495a-a9ab-caef9be3fb08,2026-05-11T14:52:41.967748+00:00,postal_code,primaria,privadas,/,NaN,21.101535,-101.583732,1500,...,primaria,desktop_list,map_results,https://www.edunautica.mx/primaria/guanajuato/...,11EPR0486U,/primaria/guanajuato/leon,desktop,1777991512,4,1777991512108
637,c68fc102-9a70-495a-a9ab-caef9be3fb08,2026-05-11T14:52:41.967748+00:00,postal_code,primaria,privadas,/,NaN,21.101535,-101.583732,1500,...,primaria,desktop_list,map_results,https://www.edunautica.mx/primaria/guanajuato/...,11DPR2770E,/primaria/guanajuato/leon,desktop,1777991512,4,1777991512108
638,c68fc102-9a70-495a-a9ab-caef9be3fb08,2026-05-11T14:52:41.967748+00:00,postal_code,primaria,privadas,/,NaN,21.101535,-101.583732,1500,...,primaria,desktop_list,map_results,https://www.edunautica.mx/primaria/guanajuato/...,11DPR0702U,/primaria/guanajuato/leon,desktop,1777991512,4,1777991512108
639,c68fc102-9a70-495a-a9ab-caef9be3fb08,2026-05-11T14:52:41.967748+00:00,postal_code,primaria,privadas,/,NaN,21.101535,-101.583732,1500,...,primaria,desktop_list,map_results,https://www.edunautica.mx/primaria/guanajuato/...,11EPR0700V,/primaria/guanajuato/leon,desktop,1777991512,4,1777991512108
640,c68fc102-9a70-495a-a9ab-caef9be3fb08,2026-05-11T14:52:41.967748+00:00,postal_code,primaria,privadas,/,NaN,21.101535,-101.583732,1500,...,primaria,desktop_list,map_results,https://www.edunautica.mx/primaria/guanajuato/...,11EPR0486U,/primaria/guanajuato/leon,desktop,1777991512,4,1777991512108
641,c68fc102-9a70-495a-a9ab-caef9be3fb08,2026-05-11T14:52:41.967748+00:00,postal_code,primaria,privadas,/,NaN,21.101535,-101.583732,1500,...,primaria,desktop_list,map_results,https://www.edunautica.mx/primaria/guanajuato/...,11DPR2770E,/primaria/guanajuato/leon,desktop,1777991512,4,1777991512108


## ETL de LogDrains de Vercel

Este es solo un pequeño analisis exploratorio para comprender como estaban estructurados los logs drains desde vercel. En este punto se sigue evaluando la exploración de datos para comprender como abordar el probelma.

In [13]:
parquet_route = VERCEL_DRAINS / 'sample_20260528.parquet'
log_data = pd.read_parquet(parquet_route)

agents_data = log_data.copy()
agents_data.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 103969 entries, 0 to 103968
Data columns (total 35 columns):
 #   Column                 Non-Null Count   Dtype  
---  ------                 --------------   -----  
 0   id                     103969 non-null  object 
 1   host                   103969 non-null  object 
 2   path                   103969 non-null  object 
 3   level                  103969 non-null  object 
 4   branch                 103969 non-null  object 
 5   source                 103969 non-null  object 
 6   ja4Digest              94813 non-null   object 
 7   projectId              103969 non-null  object 
 8   requestId              103969 non-null  object 
 9   timestamp              103969 non-null  int64  
 10  environment            103969 non-null  object 
 11  projectName            103969 non-null  object 
 12  deploymentId           103969 non-null  object 
 13  executionRegion        103969 non-null  object 
 14  proxy.host             103969 non-nu

In [14]:
agents_data['message'].unique()

array([None,
       'START RequestId: 2d81f5da-064e-4a32-b2ce-2c89717f568b\n[GET] /primaria/mexico/cuautitlan/instituto-luis-gonzalez-y-gonzalez-15ppr7049r status=200\nEND RequestId: 2d81f5da-064e-4a32-b2ce-2c89717f568b\nREPORT RequestId: 2d81f5da-064e-4a32-b2ce-2c89717f568b Duration: 934 ms Billed Duration: 934 ms Memory Size: 1769 MB Max Memory Used: 327 MB',
       'START RequestId: cf09c638-916e-490c-ae11-259454be8f28\n[GET] /secundaria/cdmx/iztapalapa/escuela-secundaria-tecnica-87-09dst0087v status=200\nEND RequestId: cf09c638-916e-490c-ae11-259454be8f28\nREPORT RequestId: cf09c638-916e-490c-ae11-259454be8f28 Duration: 739 ms Billed Duration: 739 ms Memory Size: 1769 MB Max Memory Used: 330 MB',
       ...,
       'START RequestId: 26753595-afb9-400c-ac25-08f3d3ff48a7\n[GET] /primaria/yucatan/merida/carlos-fuentes-31dpr2083c status=200\nEND RequestId: 26753595-afb9-400c-ac25-08f3d3ff48a7\nREPORT RequestId: 26753595-afb9-400c-ac25-08f3d3ff48a7 Duration: 534 ms Billed Duration: 534 

In [15]:
cols_to_drop = [
    'id','host', 'level', 'branch', 'source',
    'projectId', 'environment', 'projectName',
    'deploymentId', 'executionRegion', 'proxy.host',
    'proxy.region', 'proxy.scheme', 'proxy.cacheId',
    'proxy.pathType', 'proxy.vercelId', 'proxy.vercelCache',
    'proxy.lambdaRegion', 'type', 'instanceId', 'statusCode', 'invocationId','proxy.pathTypeVariant'
]

agents_clean_data = agents_data.drop(columns = cols_to_drop).copy()
agents_clean_data['proxy.userAgent'] = agents_clean_data['proxy.userAgent'].apply(
    lambda x: x[0] if isinstance(x, list) and len(x) > 0 else x
)

agents_clean_data.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 103969 entries, 0 to 103968
Data columns (total 12 columns):
 #   Column            Non-Null Count   Dtype  
---  ------            --------------   -----  
 0   path              103969 non-null  object 
 1   ja4Digest         94813 non-null   object 
 2   requestId         103969 non-null  object 
 3   timestamp         103969 non-null  int64  
 4   proxy.path        103969 non-null  object 
 5   proxy.method      103969 non-null  object 
 6   proxy.referer     18218 non-null   object 
 7   proxy.clientIp    103969 non-null  object 
 8   proxy.timestamp   103969 non-null  int64  
 9   proxy.userAgent   103969 non-null  object 
 10  proxy.statusCode  103933 non-null  float64
 11  message           9156 non-null    object 
dtypes: float64(1), int64(2), object(9)
memory usage: 9.5+ MB


In [16]:
agents_clean_data.to_parquet(os.path.join(str(PROCESSED_ROUTE), 'log_drains_processed.parquet'))

In [17]:
agents_clean_data

,path,ja4Digest,requestId,timestamp,proxy.path,proxy.method,proxy.referer,proxy.clientIp,proxy.timestamp,proxy.userAgent,proxy.statusCode,message
0,[niveles]/[entidades]/[municipios]/[escuelas],t13d1516h2_8daaf6152771_d8a2da3f94cd,9c6cx-1780024605788-c20e879d285d,1780024605952,/secundaria/mexico/chimalhuacan/ofic-no-0534-t...,GET,https://www.google.com/,162.120.186.163,1780024605788,[Mozilla/5.0 (Linux; Android 10; K) AppleWebKi...,200.0,None
1,[niveles]/[entidades]/[municipios]/[escuelas],t13d181300_e8a523a41297_43ade6aba3df,f4ftp-1780024609032-10b8f96c14ab,1780024609238,/preparatoria/coahuila/torreon/colegio-america...,GET,None,66.249.79.226,1780024609032,[Mozilla/5.0 (Linux; Android 6.0.1; Nexus 5X B...,200.0,None
2,[niveles]/[entidades]/[municipios],t13d1516h2_8daaf6152771_d8a2da3f94cd,r2gfb-1780024610295-df2d4caf7a05,1780024610441,/primaria/guanajuato/san-miguel-de-allende,GET,https://www.bing.com/,187.190.202.69,1780024610295,[Mozilla/5.0 (Windows NT 10.0; Win64; x64) App...,200.0,None
3,/[niveles]/[entidades]/[municipios]/[escuelas],None,p2c8n-1780024617471-3907e8a7f8f8,1780024617610,/primaria/mexico/cuautitlan/instituto-luis-gon...,GET,https://www.google.com/,187.189.87.84,1780024617471,[Mozilla/5.0 (iPhone; CPU iPhone OS 26_5_0 lik...,200.0,START RequestId: 2d81f5da-064e-4a32-b2ce-2c897...
4,[niveles]/[entidades]/[municipios]/publicas,t13d1011h2_61a7ad8aa9b6_3fcd1a44f3e3,ts6sz-1780024620377-165798b2e77d,1780024620435,/secundaria/zacatecas/miguel-auza/publicas,GET,https://www.edunautica.mx/secundaria/zacatecas...,74.7.227.187,1780024620377,"[Mozilla/5.0 AppleWebKit/537.36 (KHTML, like G...",200.0,None
...,...,...,...,...,...,...,...,...,...,...,...,...
103964,[niveles]/[entidades]/[municipios]/[escuelas],t13d1011h2_61a7ad8aa9b6_3fcd1a44f3e3,d9cvr-1780002135435-2fa4c063ea90,1780002135578,/primaria/veracruz/alamo-temapache/francisco-i...,GET,None,216.73.216.150,1780002135435,"[Mozilla/5.0 AppleWebKit/537.36 (KHTML, like G...",200.0,None
103965,[niveles]/[entidades]/[municipios]/[escuelas],t13d1011h2_61a7ad8aa9b6_3fcd1a44f3e3,qsc9c-1780002132871-4699d9fdc026,1780002133068,/primaria/veracruz/xalapa/manuel-c-tello-30dpr...,GET,None,216.73.216.150,1780002132871,"[Mozilla/5.0 AppleWebKit/537.36 (KHTML, like G...",200.0,None
103966,cct,t13d1011h2_61a7ad8aa9b6_3fcd1a44f3e3,vbtb5-1780002139044-fdb373ec915a,1780002139055,/cct?cct=12ETK0176I&nivel=preparatoria,GET,None,216.73.216.150,1780002139044,"[Mozilla/5.0 AppleWebKit/537.36 (KHTML, like G...",200.0,None
103967,cct,t13d1011h2_61a7ad8aa9b6_3fcd1a44f3e3,db5kc-1780002136899-f624aa545bf4,1780002136905,/cct?cct=13DTV0448P&nivel=secundaria,GET,None,216.73.216.150,1780002136899,"[Mozilla/5.0 AppleWebKit/537.36 (KHTML, like G...",200.0,None
